In [ ]:
import os
from mistralai import Mistral
import os
import json
import random
import time

In [3]:
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
#kostenlloses model
model =  "open-mistral-nemo"
client = Mistral(api_key=api_key)

In [8]:
# File paths
input_path = r"C:\Users\cdoering\Downloads\filtered_output.json"
output_dir = "ontology_output"
os.makedirs(output_dir, exist_ok=True)
output_file_path = os.path.join(output_dir, "ontology_try_chiara.json")

In [9]:

# Ontology generation prompt (system role)
ontology_system_prompt = """
You are an expert legal ontology engineer.

Your task is to extract ontology components from the given legal ruling text (German court decisions). 
Identify and return three structured components:

1. **Classes**: Legal entities or concepts (e.g., Urteil, Anspruch, Person, Tatbestand, Einrede)
2. **Properties**: Verbs or phrases representing relationships or attributes (e.g., beinhaltet, basiert auf, verhindert)
3. **Relationships**: Triples (Subject)-(Predicate)-(Object), connecting two classes using a property

Requirements:
- Only extract concepts if they have relationships.
- Use clear legal language in German.
- Output must follow this structure (example):

{
  "classes": ["Urteil", "Anspruch", "Tatbestand"],
  "properties": ["beinhaltet", "basiert auf"],
  "relationships": [
    ["Urteil", "beinhaltet", "Tatbestand"],
    ["Anspruch", "basiert auf", "Tatbestand"]
  ]
}

Only return JSON. Do not explain or comment.
"""

# Function to build user prompt for ontology extraction
def build_ontology_prompt(text):
    return f"Extract ontology components from the following legal text:\n\n{text[:10000]}"

# Load documents
with open(input_path, "r", encoding="utf-8") as f:
    documents = json.load(f)

# Process a sample of 20 documents
random.seed(42)
sampled_documents = dict(random.sample(list(documents.items()), 20))

ontology_results = {}

# Main loop
for i, (doc_id, doc_data) in enumerate(sampled_documents.items()):
    text_data = doc_data.get("text", {}).get("entscheidungsinhalt", {})
    gruende_list = text_data.get("gruende", {}).get("gruende", [])

    if not gruende_list:
        print(f"Skipping {doc_id} (no 'gruende')")
        continue

    text = "\n".join(gruende_list).strip()
    user_prompt = build_ontology_prompt(text)

    try:
        response = client.chat.complete(
            model=model,
            messages=[
                {"role": "system", "content": ontology_system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.5
        )

        content = response.choices[0].message.content.strip()

        # Parse response safely
        try:
            ontology = json.loads(content)
            ontology_results[doc_id] = ontology
            print(f"Ontology extracted for {doc_id}")
        except json.JSONDecodeError:
            print(f"Invalid JSON for {doc_id}, skipping.")
            continue

        time.sleep(1)

    except Exception as e:
        print(f"Error for {doc_id}: {e}")

# Save all ontology triples
with open(output_file_path, "w", encoding="utf-8") as f_out:
    json.dump(ontology_results, f_out, indent=2, ensure_ascii=False)

print("✅ Ontology extraction completed.")


Invalid JSON for 1576929.xml, skipping.
Ontology extracted for 1653713.xml
Ontology extracted for 1596591.xml
Ontology extracted for 1550917.xml
Ontology extracted for 1524814.xml
Ontology extracted for 1549422.xml
Ontology extracted for 1503264.xml
Ontology extracted for 1548573.xml
Ontology extracted for 1653040.xml
Ontology extracted for 1597921.xml
Ontology extracted for 1521870.xml
Ontology extracted for 1500675.xml
Ontology extracted for 1652739.xml
Ontology extracted for 1659170.xml
Ontology extracted for 1578989.xml
Ontology extracted for 1559224.xml
Ontology extracted for 1654866.xml
Ontology extracted for 1548575.xml
Ontology extracted for 0963480.xml
Ontology extracted for 1585808.xml
✅ Ontology extraction completed.


In [12]:
import os
import json
import random
import time
import re

# === File paths ===
input_path = r"C:\Users\cdoering\Downloads\filtered_output.json"
cq_path = r"C:\Users\cdoering\OneDrive - IW\Dokumente\Capstone\CapstoneDATEV25\competency_questions_output\competency_questions_all_documents.txt"
output_file_path = "ontology_output/ontology_with_cq.json"
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

# === Load documents ===
with open(input_path, "r", encoding="utf-8") as f:
    documents = json.load(f)

# === Load competency questions ===
cq_by_doc_id = {}
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()

cq_blocks = re.split(r'Dokument:\s*', content)
for block in cq_blocks[1:]:
    lines = block.strip().splitlines()
    doc_id = lines[0].strip()
    frage = next((l for l in lines if l.lower().startswith("frage:")), None)
    quelle = next((l for l in lines if "quelle" in l.lower()), None)
    if frage and quelle:
        cq_by_doc_id[doc_id] = {
            "question": frage.replace("Frage:", "").strip(),
            "source": quelle.replace("Quelle:", "").strip()
        }

# === Use same sampling logic as before ===
random.seed(100)
sampled_documents = dict(random.sample(list(documents.items()), 21))

# === Ontology system prompt ===
ontology_system_prompt = """
You are an expert legal ontology engineer.

Your task is to extract ontology components from the given legal ruling text (German court decisions). 
Identify and return three structured components:

1. **Classes**: Legal entities or concepts (e.g., Urteil, Entscheidungsgründe, Anspruch, Person, Tatbestand, Eiwendung, Rubrum, Tenor)
2. **Properties**: Verbs or phrases representing relationships or attributes (e.g., beinhaltet, basiert auf, verhindert, erörtern)
3. **Relationships**: Triples (Subject)-(Predicate)-(Object), connecting two classes using a property

Requirements:
- Only extract concepts if they have relationships.
- Use clear legal language in German.
- Output must follow this structure (example):

{
  "classes": ["Urteil", "Anspruch", "Tatbestand"],
  "properties": ["beinhaltet", "basiert auf"],
  "relationships": [
    ["Urteil", "beinhaltet", "Tatbestand"],
    ["Anspruch", "basiert auf", "Tatbestand"]
  ]
}

Only return JSON. Do not explain or comment.
"""

# === Build prompt with CQ (if available) ===
def build_ontology_prompt(text, cq_entry=None):
    base_prompt = f"Extract ontology components from the following legal text:\n\n{text[:10000]}"
    if cq_entry:
        base_prompt += f"\n\nConsider the following competency question to guide your extraction:\nFrage: {cq_entry['question']}"
    return base_prompt

# === Main loop ===
ontology_results = {}

for i, (doc_id, doc_data) in enumerate(sampled_documents.items()):
    text_data = doc_data.get("text", {}).get("entscheidungsinhalt", {})
    gruende_list = text_data.get("gruende", {}).get("gruende", [])

    if not gruende_list:
        print(f"⚠️ Skipping {doc_id} (no 'gruende')")
        continue

    text = "\n".join(gruende_list).strip()
    cq_entry = cq_by_doc_id.get(doc_id)
    user_prompt = build_ontology_prompt(text, cq_entry)

    try:
        response = client.chat.complete(
            model=model,
            messages=[
                {"role": "system", "content": ontology_system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.5
        )

        content = response.choices[0].message.content.strip()

        try:
            ontology = json.loads(content)
            ontology_results[doc_id] = {
                "ontology": ontology,
                "competency_question": cq_entry
            }
            print(f"✅ Ontology extracted for {doc_id}")
        except json.JSONDecodeError:
            print(f"❌ Invalid JSON for {doc_id}, skipping.")

        time.sleep(1)

    except Exception as e:
        print(f"❌ Error for {doc_id}: {e}")

# === Save results ===
with open(output_file_path, "w", encoding="utf-8") as f_out:
    json.dump(ontology_results, f_out, indent=2, ensure_ascii=False)

print("✅ Ontology extraction with competency questions completed.")


✅ Ontology extracted for 1548765.xml
✅ Ontology extracted for 1549417.xml
✅ Ontology extracted for 1567715.xml
✅ Ontology extracted for 1658302.xml
✅ Ontology extracted for 1569756.xml
⚠️ Skipping 1681077.xml (no 'gruende')
✅ Ontology extracted for 1536966.xml
✅ Ontology extracted for 1525185.xml
✅ Ontology extracted for 0960919.xml
✅ Ontology extracted for 1541664.xml
⚠️ Skipping 1682044.xml (no 'gruende')
✅ Ontology extracted for 1570226.xml
✅ Ontology extracted for 1521301.xml
✅ Ontology extracted for 1596648.xml
✅ Ontology extracted for 1544176.xml
✅ Ontology extracted for 1590874.xml
✅ Ontology extracted for 1526301.xml
✅ Ontology extracted for 1657134.xml
✅ Ontology extracted for 1532441.xml
✅ Ontology extracted for 1568706.xml
✅ Ontology extracted for 1535555.xml
✅ Ontology extraction with competency questions completed.
